# matmul-2d — worked example 1: Verify Matrix Multiplication Against the Identity Matrix

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `matmul-2d`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The rule for 2-D matrix multiplication is: `(M, K) @ (K, N) → (M, N)`. The inner dimensions must match. Multiplying any matrix by the identity matrix of the appropriate size returns the original matrix unchanged — this is a useful sanity check for validating a matmul implementation.

## Worked solution

**Step 1 — build a rectangular matrix A of shape (3, 4).**
We create `A` with known values so we can predict the result exactly.

**Step 2 — construct compatible identity matrices.**
For `A @ I_right` we need `I_right` of shape `(4, 4)` — it contracts A's columns (K=4). For `I_left @ A` we need `I_left` of shape `(3, 3)` — it contracts A's rows (M=3).

**Step 3 — compute both products and check the shape rule.**
The output of `A @ I_right` has shape `(3, 4) @ (4, 4) → (3, 4)`. The output of `I_left @ A` has shape `(3, 3) @ (3, 4) → (3, 4)`. Both should equal `A`.

**Step 4 — verify numerically.**
We assert that `A @ I_right` and `I_left @ A` are both numerically equal to `A` using `torch.allclose`. This confirms the shape rule and the `@` operator's correctness.

In [ ]:
import torch as t

t.manual_seed(5)
A = t.randn(3, 4)  # shape (3, 4)

# Identity matrices compatible with left/right multiplication
I_right = t.eye(4)   # (4, 4) — contracts with A's column dim
I_left  = t.eye(3)   # (3, 3) — contracts with A's row dim

# Perform matmuls
out_right = A @ I_right   # (3,4)@(4,4) -> (3,4)
out_left  = I_left @ A    # (3,3)@(3,4) -> (3,4)

print(f"A shape:          {A.shape}")
print(f"A @ I_right shape: {out_right.shape}")
print(f"I_left @ A shape:  {out_left.shape}")
print(f"A @ I_right == A:  {t.allclose(out_right, A)}")
print(f"I_left @ A  == A:  {t.allclose(out_left, A)}")

# Demonstrate the shape rule function
def matmul_outshape(shape_a, shape_b):
    m, k1 = shape_a
    k2, n = shape_b
    return (m, n) if k1 == k2 else None

print(f"\nShape rule (3,4)@(4,4): {matmul_outshape((3,4),(4,4))}")
print(f"Shape rule (3,4)@(5,2): {matmul_outshape((3,4),(5,2))}  <- None (inner mismatch)")